# Unidade 1 - Bloco prático da Aula 02: métricas sob desbalanceamento

Compara um classificador trivial, uma regressão logística e uma floresta aleatória num problema binário sintético com 5% de positivos, usando seis métricas lado a lado (acurácia, precisão, revocação, F1, MCC, AUC). Semente fixa (`seed=42`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# ============================================================
# 1. CRIANDO UM PROBLEMA DESBALANCEADO
# ============================================================

# Aproximadamente 95% da classe 0
# Aproximadamente 5% da classe 1

X, y = make_classification(
    n_samples=5000,
    n_features=10,
    n_informative=5,
    weights=[0.95, 0.05],
    flip_y=0.01,
    random_state=42
)

X_tr, X_te, y_tr, y_te = train_test_split(
    X,
    y,
    test_size=0.3,
    stratify=y,
    random_state=42
)

print("=" * 80)
print("DISTRIBUIÇÃO DO CONJUNTO DE TESTE")
print("=" * 80)

n_negativos = np.sum(y_te == 0)
n_positivos = np.sum(y_te == 1)

print(f"Classe 0, negativos: {n_negativos} ({n_negativos / len(y_te):.1%})")
print(f"Classe 1, positivos: {n_positivos} ({n_positivos / len(y_te):.1%})")

# ============================================================
# 2. MODELOS
# ============================================================

modelos = {
    "Trivial, classe majoritária": DummyClassifier(
        strategy="most_frequent"
    ),

    "Regressão logística": LogisticRegression(
        max_iter=1000
    ),

    "Floresta aleatória": RandomForestClassifier(
        random_state=42
    )
}

resultados = {}

# ============================================================
# 3. TREINAMENTO E MÉTRICAS
# ============================================================

for nome, modelo in modelos.items():

    modelo.fit(X_tr, y_tr)

    pred = modelo.predict(X_te)

    if hasattr(modelo, "predict_proba"):
        score = modelo.predict_proba(X_te)[:, 1]
        auc = roc_auc_score(y_te, score)
    else:
        score = pred
        auc = roc_auc_score(y_te, score)

    matriz = confusion_matrix(y_te, pred)

    tn, fp, fn, tp = matriz.ravel()

    resultados[nome] = {
        "modelo": modelo,
        "pred": pred,
        "matriz": matriz,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "accuracy": accuracy_score(y_te, pred),
        "precision": precision_score(y_te, pred, zero_division=0),
        "recall": recall_score(y_te, pred, zero_division=0),
        "f1": f1_score(y_te, pred, zero_division=0),
        "mcc": matthews_corrcoef(y_te, pred),
        "auc": auc
    }

# ============================================================
# 4. TABELA COM RESULTADOS
# ============================================================

print("\n" + "=" * 100)
print("RESULTADOS DOS MODELOS")
print("=" * 100)

print(
    f"{'Modelo':<32}"
    f"{'Acur.':>8}"
    f"{'Prec.':>8}"
    f"{'Recall':>8}"
    f"{'F1':>8}"
    f"{'MCC':>8}"
    f"{'AUC':>8}"
)

print("-" * 100)

for nome, r in resultados.items():

    print(
        f"{nome:<32}"
        f"{r['accuracy']:>8.3f}"
        f"{r['precision']:>8.3f}"
        f"{r['recall']:>8.3f}"
        f"{r['f1']:>8.3f}"
        f"{r['mcc']:>8.3f}"
        f"{r['auc']:>8.3f}"
    )

# ============================================================
# 5. MATRIZES DE CONFUSÃO
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (nome, r) in zip(axes, resultados.items()):

    disp = ConfusionMatrixDisplay(
        confusion_matrix=r["matriz"],
        display_labels=["Classe 0", "Classe 1"]
    )

    disp.plot(
        ax=ax,
        colorbar=False,
        values_format="d"
    )

    ax.set_title(nome)

plt.suptitle(
    "Comparação das Matrizes de Confusão",
    fontsize=16
)

plt.tight_layout()
plt.show()

# ============================================================
# 6. EXPLICAÇÃO DO COMPORTAMENTO DE CADA MODELO
# ============================================================

print("\n" + "=" * 100)
print("O QUE CADA MODELO ESTÁ FAZENDO?")
print("=" * 100)

for nome, r in resultados.items():

    print("\n" + "-" * 100)
    print(nome.upper())
    print("-" * 100)

    print(f"Verdadeiros negativos, TN: {r['TN']}")
    print(f"Falsos positivos, FP:     {r['FP']}")
    print(f"Falsos negativos, FN:     {r['FN']}")
    print(f"Verdadeiros positivos, TP:{r['TP']}")

    total_pred_positivos = r["TP"] + r["FP"]
    total_pred_negativos = r["TN"] + r["FN"]

    print()
    print(f"Predições como classe 0: {total_pred_negativos}")
    print(f"Predições como classe 1: {total_pred_positivos}")

    print("\nInterpretação:")

    # Modelo que nunca prevê positivo
    if total_pred_positivos == 0:

        print(
            "→ Este modelo NÃO identificou nenhum exemplo como classe positiva."
        )

        print(
            "→ Ele simplesmente classificou tudo como classe 0."
        )

        print(
            f"→ Mesmo assim conseguiu {r['accuracy']:.1%} de acurácia."
        )

        print(
            "→ Isso acontece porque aproximadamente 95% dos exemplos já são da classe 0."
        )

        print(
            "→ Portanto, a acurácia parece boa, mas o modelo é inútil para detectar a classe rara."
        )

        print(
            "→ Recall = 0, F1 = 0 e MCC próximo de 0 deixam esse problema evidente."
        )

    else:

        print(
            f"→ O modelo encontrou {r['TP']} dos "
            f"{r['TP'] + r['FN']} positivos reais."
        )

        print(
            f"→ Isso corresponde a um recall de {r['recall']:.1%}."
        )

        print(
            f"→ Das {total_pred_positivos} vezes em que disse 'positivo', "
            f"{r['TP']} estavam corretas."
        )

        print(
            f"→ Isso corresponde a uma precisão de {r['precision']:.1%}."
        )

        if r["recall"] < 0.50:
            print(
                "→ O modelo ainda deixa escapar muitos positivos, ou seja, possui muitos falsos negativos."
            )

        elif r["recall"] < 0.80:
            print(
                "→ O modelo consegue detectar boa parte dos positivos, mas ainda perde alguns casos."
            )

        else:
            print(
                "→ O modelo consegue detectar a maior parte da classe positiva."
            )

        if r["mcc"] > 0.7:
            print(
                "→ O MCC indica uma classificação globalmente muito consistente."
            )

        elif r["mcc"] > 0.4:
            print(
                "→ O MCC indica uma relação razoável entre as previsões e as classes reais."
            )

        elif r["mcc"] > 0:
            print(
                "→ O MCC mostra que existe algum poder preditivo, mas ainda limitado."
            )

        else:
            print(
                "→ O MCC indica que o modelo praticamente não apresenta poder preditivo."
            )

# ============================================================
# 7. RESUMO DIDÁTICO
# ============================================================

print("\n" + "=" * 100)
print("RESUMO")
print("=" * 100)

print("""
1. TRIVIAL
   Não aprende padrões.
   Ele olha qual classe é mais comum e responde sempre aquela classe.

2. REGRESSÃO LOGÍSTICA
   Aprende uma fronteira de decisão relativamente simples.
   Ela tenta encontrar uma combinação das características que separe as classes.

3. FLORESTA ALEATÓRIA
   Aprende relações mais complexas e não lineares.
   Várias árvores tomam decisões e depois combinam seus resultados.

IMPORTANTE

Em problemas desbalanceados, uma acurácia de 95% pode ser completamente enganosa.

Se 95% dos exemplos são negativos, um modelo pode responder "negativo" para
todo mundo e conseguir aproximadamente 95% de acurácia sem detectar nenhum
positivo.

Por isso, devemos observar principalmente:

Recall     → quantos positivos reais foram encontrados
Precisão   → quantos positivos previstos realmente eram positivos
F1         → equilíbrio entre precisão e recall
MCC        → qualidade geral da classificação, considerando toda a matriz
AUC        → capacidade de separar/rankear positivos e negativos
""")

DISTRIBUIÇÃO DO CONJUNTO DE TESTE
Classe 0, negativos: 1418 (94.5%)
Classe 1, positivos: 82 (5.5%)
